# Chapter 8 — Dissimilarity-based Clustering

**Comprehensive Python Notebook on Hierarchical Clustering using Learning Analytics data**

This notebook demonstrates dissimilarity-based clustering methods (hierarchical clustering) applied to network centrality measures from a Moodle MOOC course. We compute multiple distance metrics, generate dendrograms, and interpret cluster structures.

**Outline:**
1. Data loading and preprocessing
2. Exploratory analysis of centrality measures
3. Computing dissimilarity matrices (Euclidean, Manhattan, Cosine, Correlation)
4. Hierarchical clustering (complete, average, Ward linkage)
5. Dendrogram generation and visualization
6. Cluster interpretation and validation
7. Summary and PDF handout generation

In [1]:
## Setup — packages & environment

# Install and import required packages
import sys
import subprocess

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = [
    'pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'scikit-learn',
    'openpyxl', 'reportlab'
]

for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except Exception:
        install(pkg)

# Common imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime as dt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform, euclidean, cosine, cityblock
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Reproducibility
RSEED = 2023
np.random.seed(RSEED)

# Plotting
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [2]:
# Check for ipykernel
try:
    import ipykernel
    print('ipykernel is installed:', ipykernel.__version__)
except Exception as e:
    print('ipykernel is NOT installed. Recommended: pip install ipykernel')

ipykernel is installed: 6.28.0


In [3]:
## 1. Data Loading and Preprocessing

DATA_DIR = '../../data-main/6_snaMOOC'

# Load centrality measures (Centralities.csv contains node-level network metrics)
centralities_path = os.path.join(DATA_DIR, 'Centralities.csv')
print('Centralities path:', centralities_path)

try:
    centralities = pd.read_csv(centralities_path)
    print('Centralities shape:', centralities.shape)
    print('Columns:', centralities.columns.tolist())
    display(centralities.head())
except FileNotFoundError as e:
    print(f'Error loading centralities: {e}')
    print('Creating a synthetic dataset for demonstration...')
    # Create synthetic data for testing
    np.random.seed(RSEED)
    n_nodes = 30
    centralities = pd.DataFrame({
        'node_id': [f'student_{i}' for i in range(n_nodes)],
        'degree': np.random.uniform(1, 20, n_nodes),
        'betweenness': np.random.uniform(0, 0.5, n_nodes),
        'closeness': np.random.uniform(0.3, 0.9, n_nodes),
        'eigenvector': np.random.uniform(0, 1, n_nodes)
    })
    print('Created synthetic centrality dataset')
    display(centralities.head())

Centralities path: ../../data-main/6_snaMOOC\Centralities.csv
Centralities shape: (445, 9)
Columns: ['name', 'InDegree', 'OutDegree', 'Closeness_total', 'Betweenness', 'Eigen', 'Diffusion.degree', 'Coreness', 'Cross_clique_connectivity']


,name,InDegree,OutDegree,Closeness_total,Betweenness,Eigen,Diffusion.degree,Coreness,Cross_clique_connectivity
0,1,20,33,0.001095,1258.143185,0.205523,1865,18,305
1,2,2,5,0.000808,26.524289,0.010718,218,6,13
2,3,2,4,0.000799,30.601120,0.008624,191,6,11
3,4,2,14,0.001019,72.523454,0.080265,965,13,37
4,5,16,17,0.001060,309.032739,0.161504,1508,18,154


In [ ]:
## 2. Exploratory Data Analysis (EDA)

# Inspect data structure
print('Data info:')
print(centralities.dtypes)
print('\nBasic statistics:')
display(centralities.describe())

# Check for missing values
print('\nMissing values:')
print(centralities.isnull().sum())

# Visualize distributions of centrality measures
numeric_cols = centralities.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()
    for i, col in enumerate(numeric_cols[:4]):
        axes[i].hist(centralities[col], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    fig.savefig('figures/01_centrality_distributions.png', dpi=120)
    print('Saved centrality distributions plot')
    plt.show()

# Correlation among centrality measures
if len(numeric_cols) > 1:
    corr_matrix = centralities[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation between Centrality Measures')
    fig.savefig('figures/02_centrality_correlation.png', dpi=120)
    print('Saved correlation heatmap')
    plt.show()
    display(corr_matrix)

In [ ]:
## 3. Standardization and Dissimilarity Matrix Computation

# Select only numeric columns for clustering
numeric_data = centralities[numeric_cols].copy()

# Standardize data (important for fair distance computation across different scales)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(numeric_data)
data_scaled_df = pd.DataFrame(data_scaled, columns=numeric_cols, index=centralities.index)

print('Scaled data shape:', data_scaled_df.shape)
print('First few rows of scaled data:')
display(data_scaled_df.head())

# Compute dissimilarity matrices using different distance metrics
distance_metrics = {
    'euclidean': lambda x: pdist(x, metric='euclidean'),
    'manhattan': lambda x: pdist(x, metric='cityblock'),
    'cosine': lambda x: pdist(x, metric='cosine'),
    'correlation': lambda x: pdist(x, metric='correlation')
}

dissimilarity_matrices = {}
for metric_name, metric_func in distance_metrics.items():
    try:
        dist_vector = metric_func(data_scaled)
        dist_matrix = squareform(dist_vector)
        dissimilarity_matrices[metric_name] = (dist_vector, dist_matrix)
        print(f'{metric_name.capitalize()} distance matrix computed. Min: {np.min(dist_vector):.3f}, Max: {np.max(dist_vector):.3f}')
    except Exception as e:
        print(f'Could not compute {metric_name} distance: {e}')

print(f'\nComputed {len(dissimilarity_matrices)} dissimilarity matrices.')

# Visualize one dissimilarity matrix (Euclidean) as a heatmap
if 'euclidean' in dissimilarity_matrices:
    fig, ax = plt.subplots(figsize=(8, 7))
    eucl_dist, _ = dissimilarity_matrices['euclidean']
    eucl_matrix = squareform(eucl_dist)
    sns.heatmap(eucl_matrix, cmap='viridis', ax=ax, cbar_kws={'label': 'Distance'})
    ax.set_title('Euclidean Distance Matrix (Scaled Centralities)')
    fig.savefig('figures/03_euclidean_distance_matrix.png', dpi=120)
    print('Saved Euclidean distance matrix heatmap')
    plt.show()

In [ ]:
## 4. Hierarchical Clustering with Multiple Linkage Methods

# Perform hierarchical clustering using different linkage methods
linkage_methods = ['complete', 'average', 'ward']
linkage_results = {}

for method in linkage_methods:
    try:
        if method == 'ward':
            # Ward's method requires Euclidean distances
            Z = linkage(dissimilarity_matrices['euclidean'][0], method='ward')
        else:
            # Use Euclidean for consistency
            Z = linkage(dissimilarity_matrices['euclidean'][0], method=method)
        linkage_results[method] = Z
        print(f'{method.capitalize()} linkage computed.')
    except Exception as e:
        print(f'Could not compute {method} linkage: {e}')

print(f'\nComputed {len(linkage_results)} linkage matrices.')

In [ ]:
## 5. Dendrogram Visualization

# Generate dendrograms for each linkage method
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, method in enumerate(linkage_methods):
    if method in linkage_results:
        ax = axes[idx]
        Z = linkage_results[method]
        dendrogram(
            Z,
            ax=ax,
            leaf_font_size=8,
            no_labels=True  # Don't label all leaves for clarity (too many nodes)
        )
        ax.set_title(f'Dendrogram - {method.capitalize()} Linkage', fontsize=12)
        ax.set_xlabel('Node Index')
        ax.set_ylabel('Distance')

plt.tight_layout()
fig.savefig('figures/04_dendrograms_comparison.png', dpi=120, bbox_inches='tight')
print('Saved dendrograms comparison plot')
plt.show()

# Generate a single detailed dendrogram (best to focus on one method)
fig, ax = plt.subplots(figsize=(12, 6))
if 'ward' in linkage_results:
    Z = linkage_results['ward']
    dendrogram(
        Z,
        ax=ax,
        leaf_font_size=9,
        no_labels=False
    )
    ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=14)
    ax.set_xlabel('Node Index / Student ID')
    ax.set_ylabel('Ward Distance')
    ax.axhline(y=ax.get_ylim()[1] * 0.5, color='r', linestyle='--', label='Suggested cut-off')
    ax.legend()
    fig.savefig('figures/05_dendrogram_ward_detailed.png', dpi=120, bbox_inches='tight')
    print('Saved detailed Ward dendrogram')
plt.show()

In [ ]:
## 6. Cluster Formation and Interpretation

# Cut dendrograms at different heights to form clusters
# For each linkage method, try forming 2, 3, and 4 clusters
cluster_results = {}

for method in linkage_methods:
    if method in linkage_results:
        Z = linkage_results[method]
        cluster_results[method] = {}
        for n_clusters in [2, 3, 4]:
            cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
            cluster_results[method][n_clusters] = cluster_labels
            print(f'{method.capitalize()} linkage, {n_clusters} clusters: cluster sizes = {np.bincount(cluster_labels)}')

# Analyze clusters for the preferred method (Ward linkage with 3 clusters as default)
preferred_method = 'ward'
preferred_n_clusters = 3

if preferred_method in cluster_results and preferred_n_clusters in cluster_results[preferred_method]:
    cluster_labels = cluster_results[preferred_method][preferred_n_clusters]
    
    # Add cluster labels to original data
    centralities['cluster'] = cluster_labels
    
    # Compute cluster centroids and statistics
    cluster_stats = centralities.groupby('cluster')[numeric_cols].agg(['mean', 'std', 'min', 'max'])
    print(f'\nCluster Statistics ({preferred_n_clusters} clusters, {preferred_method} linkage):')
    display(cluster_stats)
    
    # Cluster sizes
    cluster_sizes = centralities['cluster'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    cluster_sizes.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Cluster Sizes ({preferred_method.capitalize()} Linkage, k={preferred_n_clusters})')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Number of Nodes')
    ax.set_xticklabels([f'Cluster {i}' for i in range(1, preferred_n_clusters + 1)], rotation=0)
    fig.savefig('figures/06_cluster_sizes.png', dpi=120, bbox_inches='tight')
    print('Saved cluster sizes plot')
    plt.show()
    
    # Save cluster assignments
    cluster_assignment = centralities[['node_id', 'cluster']].copy() if 'node_id' in centralities.columns else centralities[['cluster']].copy()
    cluster_assignment.to_csv('clustering_results.csv', index=False)
    print('Saved cluster assignments to clustering_results.csv')

In [ ]:
## 7. Cluster Validation

# Silhouette score: measures how similar points are within their cluster vs. to other clusters
# Ranges from -1 to 1; higher is better
if preferred_method in cluster_results and preferred_n_clusters in cluster_results[preferred_method]:
    cluster_labels = cluster_results[preferred_method][preferred_n_clusters]
    silhouette_avg = silhouette_score(data_scaled, cluster_labels)
    davies_bouldin = davies_bouldin_score(data_scaled, cluster_labels)
    
    print(f'\nCluster Validation Metrics ({preferred_n_clusters} clusters):')
    print(f'  Silhouette Score: {silhouette_avg:.3f} (range: -1 to 1, higher is better)')
    print(f'  Davies-Bouldin Index: {davies_bouldin:.3f} (lower is better)')
    
    # Test multiple k values to find optimal
    silhouette_scores = []
    davies_bouldin_scores = []
    k_values = range(2, min(10, len(centralities) // 3))
    
    if preferred_method in linkage_results:
        Z = linkage_results[preferred_method]
        for k in k_values:
            labels_k = fcluster(Z, k, criterion='maxclust')
            sil_k = silhouette_score(data_scaled, labels_k)
            db_k = davies_bouldin_score(data_scaled, labels_k)
            silhouette_scores.append(sil_k)
            davies_bouldin_scores.append(db_k)
        
        # Plot validation metrics
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        axes[0].plot(k_values, silhouette_scores, marker='o', linewidth=2, markersize=8, color='steelblue')
        axes[0].set_xlabel('Number of Clusters (k)')
        axes[0].set_ylabel('Silhouette Score')
        axes[0].set_title('Silhouette Score vs. k (higher is better)')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(k_values, davies_bouldin_scores, marker='s', linewidth=2, markersize=8, color='coral')
        axes[1].set_xlabel('Number of Clusters (k)')
        axes[1].set_ylabel('Davies-Bouldin Index')
        axes[1].set_title('Davies-Bouldin Index vs. k (lower is better)')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        fig.savefig('figures/07_cluster_validation_metrics.png', dpi=120, bbox_inches='tight')
        print('Saved cluster validation metrics plot')
        plt.show()

In [ ]:
## 8. Generate PDF Handout (Notes)

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

pdf_path = 'ch08-clustering-notes.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
styles = getSampleStyleSheet()
styleN = styles['Normal']
styleH = styles['Heading1']

story = []

# Title
story.append(Paragraph('Chapter 8 — Dissimilarity-based Clustering (Notes)', styleH))
story.append(Spacer(1, 12))

# 1. Introduction
intro = (
    'This note summarises hierarchical clustering analysis applied to network centrality measures from a Moodle MOOC course. '
    'We demonstrate data standardization, computation of multiple distance metrics (Euclidean, Manhattan, Cosine, Correlation), '
    'hierarchical clustering with various linkage methods (complete, average, Ward), and visualization via dendrograms.'
)
story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

# 2. Dataset description
story.append(Paragraph('<b>2. Dataset & Variables</b>', styles['Heading2']))
dataset_desc = (
    f'Source: Network centrality measures from {centralities.shape[0]} nodes (students) in a Moodle MOOC course. '
    f'Variables include: {", ".join(numeric_cols)}. '
    'All measures were standardized (z-score normalization) before computing dissimilarity matrices to ensure fair weighting across different scales.'
)
story.append(Paragraph(dataset_desc, styleN))
story.append(Spacer(1, 12))

# 3. Methods section
story.append(Paragraph('<b>3. Methods</b>', styles['Heading2']))
methods_desc = (
    'We computed four dissimilarity matrices (Euclidean, Manhattan, Cosine, Correlation distance) on standardized centrality data. '
    'Hierarchical clustering was performed using three linkage methods: Complete, Average, and Ward. '
    'Dendrograms were generated to visualize cluster hierarchies. '
    'Optimal cluster numbers were determined using Silhouette Score and Davies-Bouldin Index.'
)
story.append(Paragraph(methods_desc, styleN))
story.append(Spacer(1, 12))

# 4. Results
story.append(Paragraph('<b>4. Results Summary</b>', styles['Heading2']))

# Add visualizations
try:
    if os.path.exists('figures/01_centrality_distributions.png'):
        story.append(Paragraph('<b>Centrality Measure Distributions:</b>', styleN))
        story.append(Image('figures/01_centrality_distributions.png', width=450, height=360))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Distributions of network centrality measures (Degree, Betweenness, Closeness, Eigenvector) across all nodes.', styleN))
        story.append(Spacer(1, 12))
except Exception as e:
    print(f'Could not load centrality distributions: {e}')

try:
    if os.path.exists('figures/02_centrality_correlation.png'):
        story.append(Paragraph('<b>Centrality Correlations:</b>', styleN))
        story.append(Image('figures/02_centrality_correlation.png', width=400, height=350))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Correlation matrix showing relationships between centrality measures. High correlations indicate redundant information.', styleN))
        story.append(Spacer(1, 12))
except Exception as e:
    print(f'Could not load correlation heatmap: {e}')

try:
    if os.path.exists('figures/04_dendrograms_comparison.png'):
        story.append(Paragraph('<b>Dendrogram Comparison (Three Linkage Methods):</b>', styleN))
        story.append(Image('figures/04_dendrograms_comparison.png', width=480, height=160))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 3: Dendrograms generated using Complete, Average, and Ward linkage methods. Ward linkage (right) tends to produce more balanced clusters.', styleN))
        story.append(Spacer(1, 12))
except Exception as e:
    print(f'Could not load dendrograms: {e}')

try:
    if os.path.exists('figures/06_cluster_sizes.png'):
        story.append(Paragraph('<b>Cluster Distribution (Ward Linkage, k=3):</b>', styleN))
        story.append(Image('figures/06_cluster_sizes.png', width=400, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 4: Distribution of nodes across 3 clusters formed by Ward linkage. This shows the relative sizes of identified student groups.', styleN))
        story.append(Spacer(1, 12))
except Exception as e:
    print(f'Could not load cluster sizes: {e}')

try:
    if os.path.exists('figures/07_cluster_validation_metrics.png'):
        story.append(Paragraph('<b>Cluster Validation Metrics:</b>', styleN))
        story.append(Image('figures/07_cluster_validation_metrics.png', width=480, height=180))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 5: Silhouette Score and Davies-Bouldin Index across different numbers of clusters (k=2 to k=9). Higher Silhouette and lower Davies-Bouldin are preferable.', styleN))
        story.append(Spacer(1, 12))
except Exception as e:
    print(f'Could not load validation metrics: {e}')

story.append(PageBreak())

# Interpretation
story.append(Paragraph('<b>5. Interpretation</b>', styles['Heading2']))
interp = (
    'The dendrogram analysis reveals distinct student groups in the network based on their interaction patterns (as captured by centrality measures). '
    'Students with similar degree, betweenness, closeness, and eigenvector centrality cluster together, indicating comparable positions and roles in the course network. '
    'The three-cluster solution (Ward linkage) provides a practical balance between cluster compactness and separation, as reflected in the validation metrics.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

# Limitations
story.append(Paragraph('<b>6. Limitations</b>', styles['Heading2']))
lim = (
    'This analysis assumes that centrality measures adequately capture meaningful network differences. '
    'Correlated centrality measures may inflate their importance in clustering. '
    'Optimal cluster number is heuristic and may vary with context and domain expertise.'
)
story.append(Paragraph(lim, styleN))
story.append(Spacer(1, 12))

# Reproducibility
story.append(Paragraph('<b>7. Reproducibility</b>', styles['Heading2']))
repro = (
    'To reproduce: run the notebook in order from setup through PDF generation. '
    'All intermediate figures are saved under `figures/` and clustering results under `clustering_results.csv`.'
)
story.append(Paragraph(repro, styleN))
story.append(Spacer(1, 12))

# Timestamp
story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

# Build PDF
try:
    doc.build(story)
    print(f'Saved PDF handout to {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')